# Module 3 — Lab Bài 2: Responsible AI & AI Risk Assessment
## Notebook sinh viên · Google Colab · 90 phút

**Mục tiêu:** áp dụng khung Responsible AI vào một use case cụ thể → phân loại dữ liệu → xếp mức rủi ro → lập AI Risk Assessment → chốt quyết định **go / pilot / not yet**. Không triển khai model thật; trọng tâm là **tư duy kiểm soát rủi ro**.

Notebook có sẵn phần lớn code (chỉ dùng `pandas`, không cần GPU, không gọi API, không cần internet). Sinh viên **điền nội dung** tại các ô **THỰC HÀNH** và chạy các cell để sinh bảng + markdown summary để nộp. Hai demo nhỏ (bias theo nhóm, giải thích kiểu LIME) chạy hoàn toàn trên dữ liệu mô phỏng.

| Phút | Phần |
|---|---|
| 0–10 | 0. Case mở đầu — sai 10% có giống nhau? |
| 10–20 | 1. Chọn use case + Phân loại dữ liệu |
| 20–30 | 2. Xếp mức rủi ro (EU AI Act) |
| 30–50 | 3. AI Risk Assessment 5 nhóm |
| 50–65 | 4. Explainability & Bias (2 demo) |
| 65–78 | 5. AI Use-case Canvas |
| 78–88 | 6. Quyết định go/pilot/not yet + xuất summary |
| 88–90 | 7. Peer review + nộp |

**Nộp:** notebook có output; bảng phân loại dữ liệu; bảng risk assessment đủ 5 nhóm; Canvas đầy đủ 8 ô; quyết định go/pilot/not yet kèm lý do; trả lời 4 câu debrief.

**Nhóm:** … · **Thành viên:** … · **Use case chọn:** …


# 0. Case mở đầu: sai 10% có giống nhau không?
Cùng một con số accuracy, nhưng rủi ro nghiệp vụ rất khác nhau tùy theo **hành động sau output**. Đây là điểm mở đầu để thấy: **AI chỉ là một thành phần trong quyết định triển khai**. Thiết kế tốt phải nối công nghệ với hậu quả nghiệp vụ và nghĩa vụ pháp lý.

Chạy cell dưới để nhìn cùng một `error_rate = 10%` ứng với các mức tự động hóa khác nhau.

In [1]:
import json
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

OUTPUT_DIR = Path('lab2_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

ERROR_RATE = 0.10  # cùng một con số cho cả ba
opening_case = pd.DataFrame([
    {'Tác vụ': 'Tóm tắt email',          'Hành động sau output': 'Người đọc tham khảo',    'error_rate': ERROR_RATE, 'Hậu quả khi sai': 'Khó chịu, nhưng sửa thủ công được'},
    {'Tác vụ': 'Đề xuất xử lý incident',  'Hành động sau output': 'Người ra quyết định',    'error_rate': ERROR_RATE, 'Hậu quả khi sai': 'Kỹ thuật viên đi sai hướng'},
    {'Tác vụ': 'Tự động đổi cấu hình',   'Hành động sau output': 'Hệ thống tự hành động',   'error_rate': ERROR_RATE, 'Hậu quả khi sai': 'Có thể gây sự cố diện rộng'},
])
display(opening_case)
print('=> Cùng error_rate =', ERROR_RATE, 'nhưng rủi ro nghiệp vụ tăng dần theo mức tự động hóa.')


,Tác vụ,Hành động sau output,error_rate,Hậu quả khi sai
0,Tóm tắt email,Người đọc tham khảo,0.1,"Khó chịu, nhưng sửa thủ công được"
1,Đề xuất xử lý incident,Người ra quyết định,0.1,Kỹ thuật viên đi sai hướng
2,Tự động đổi cấu hình,Hệ thống tự hành động,0.1,Có thể gây sự cố diện rộng


=> Cùng error_rate = 0.1 nhưng rủi ro nghiệp vụ tăng dần theo mức tự động hóa.


**THỰC HÀNH — điền:**
- False positive (AI báo có sự cố nhưng thực tế không) trong use case của bạn gây hậu quả gì? …
- False negative (AI bỏ sót sự cố) gây hậu quả gì? …
- Hai loại lỗi này có **tương đương** nhau không? Loại nào nguy hiểm hơn trong bối cảnh của bạn? …


# 1. Chọn use case + Phân loại dữ liệu
### Bước 1 — Chọn một use case
Chọn một trong các use case gợi ý (hoặc use case nhóm quan tâm):
1. **RAG** — trợ lý hỏi đáp sổ tay vận hành.
2. **NLP** — phân loại alarm/ticket để routing (hoặc phân tích phản hồi khách hàng).
3. **Agent** — incident assistant có tool KPI + knowledge search.
4. **CV** — object detection giám sát hạ tầng từ camera.

Cell dưới có sẵn **3 use case mẫu** (RAG / NLP ticket / Agent incident) để tham chiếu. Đổi `SELECTED_USECASE` sang use case nhóm chọn.

In [2]:
# Ba use case mẫu (mô tả ngắn để nhóm tham chiếu, không phải triển khai thật).
USECASES = {
    'RAG': {
        'ten': 'Trợ lý RAG hỏi đáp sổ tay vận hành',
        'problem': 'Kỹ thuật viên mất thời gian tìm quy trình xử lý sự cố',
        'ai_task': 'Retrieval + grounded answer',
        'data': 'Sổ tay vận hành, FAQ, incident postmortem (đã phân quyền)',
        'action': 'Người đọc tham khảo',
    },
    'NLP_TICKET': {
        'ten': 'Phân loại alarm/ticket để routing',
        'problem': 'Giảm thời gian routing ticket cho đội phù hợp',
        'ai_task': 'Classification + extraction',
        'data': '~1.000 ticket cũ, nhãn đội xử lý, severity',
        'action': 'Người ra quyết định (routing)',
    },
    'AGENT': {
        'ten': 'Incident assistant agent (tool KPI + knowledge search)',
        'problem': 'Hỗ trợ kỹ thuật viên điều tra sự cố nhanh hơn',
        'ai_task': 'Agent / tool-use + retrieval',
        'data': 'KPI realtime, log, sổ tay, lịch sử incident',
        'action': 'Hệ thống chuẩn bị hành động, người duyệt',
    },
}

# THỰC HÀNH: đổi giá trị này sang use case nhóm chọn ('RAG' | 'NLP_TICKET' | 'AGENT').
SELECTED_USECASE = 'RAG'

uc = USECASES[SELECTED_USECASE]
print('Use case đã chọn:', SELECTED_USECASE)
for k, v in uc.items():
    print(f'  - {k:10s}: {v}')


Use case đã chọn: RAG
  - ten       : Trợ lý RAG hỏi đáp sổ tay vận hành
  - problem   : Kỹ thuật viên mất thời gian tìm quy trình xử lý sự cố
  - ai_task   : Retrieval + grounded answer
  - data      : Sổ tay vận hành, FAQ, incident postmortem (đã phân quyền)
  - action    : Người đọc tham khảo


### Bước 2 — Phân loại dữ liệu đầu vào
Phân loại dữ liệu **trước**, rồi mới chọn công cụ AI phù hợp — không làm ngược lại.

| Mức | Ý nghĩa |
|---|---|
| **Public** | Tài liệu công khai, demo data |
| **Internal** | Quy trình, sổ tay, ticket đã ẩn thông tin nhạy cảm |
| **Confidential** | Cấu hình, thông tin khách hàng, báo cáo sự cố |
| **Restricted** | Credential, secret, dữ liệu cá nhân nhạy cảm |

Căn cứ pháp lý VN tham chiếu: **Nghị định 13/2023/NĐ-CP** (bảo vệ dữ liệu cá nhân — yêu cầu sự đồng ý, đánh giá tác động xử lý), **Luật An ninh mạng 2018**, **Luật An toàn thông tin mạng 2015**. *Bài học mang tính định hướng, không phải tư vấn pháp lý — luôn kiểm tra văn bản hiện hành và làm việc với bộ phận pháp chế/compliance.*

**THỰC HÀNH:** sửa `data_rows` bên dưới thành các loại dữ liệu thực của use case nhóm. Thay các ô `'…'` bằng nội dung cụ thể.

In [3]:
DATA_LEVELS = ['Public', 'Internal', 'Confidential', 'Restricted']

# THỰC HÀNH: điền bảng phân loại dữ liệu cho use case của nhóm.
# Mỗi dòng là một loại dữ liệu đầu vào. (Dưới đây là ví dụ cho RAG — hãy sửa lại.)
data_rows = [
    {'Loại dữ liệu': 'Sổ tay vận hành', 'Mức phân loại': 'Internal',
     'Có PII?': 'Không', 'Cần masking/redaction?': 'Không',
     'Cần đồng ý/đánh giá tác động?': 'Không'},
    {'Loại dữ liệu': 'Incident postmortem', 'Mức phân loại': 'Confidential',
     'Có PII?': 'Có thể', 'Cần masking/redaction?': 'Có (tên, SĐT, mã KH)',
     'Cần đồng ý/đánh giá tác động?': 'Có — rà PII theo NĐ 13/2023'},
    {'Loại dữ liệu': '…', 'Mức phân loại': '…',
     'Có PII?': '…', 'Cần masking/redaction?': '…',
     'Cần đồng ý/đánh giá tác động?': '…'},
]
data_class = pd.DataFrame(data_rows)
assert data_class['Mức phân loại'].isin(DATA_LEVELS + ['…']).all(), 'Mức phân loại phải thuộc Public/Internal/Confidential/Restricted.'
display(data_class)
data_class.to_csv(OUTPUT_DIR / 'data_classification.csv', index=False)


,Loại dữ liệu,Mức phân loại,Có PII?,Cần masking/redaction?,Cần đồng ý/đánh giá tác động?
0,Sổ tay vận hành,Internal,Không,Không,Không
1,Incident postmortem,Confidential,Có thể,"Có (tên, SĐT, mã KH)",Có — rà PII theo NĐ 13/2023
2,…,…,…,…,…


# 2. Xếp mức rủi ro (tinh thần EU AI Act)
EU AI Act phân tầng theo mức rủi ro. Mức càng cao, nghĩa vụ về đánh giá, tài liệu, giám sát con người và minh bạch càng nặng.

| Mức | Mô tả | Nghĩa vụ chính |
|---|---|---|
| **unacceptable** | Bị cấm (vd: social scoring) | Không được triển khai |
| **high** | Cho phép nhưng nhiều nghĩa vụ | Đánh giá, giám sát con người, tài liệu |
| **limited** | Nghĩa vụ minh bạch | Báo người dùng biết đang dùng AI |
| **minimal** | Phần lớn ứng dụng | Ít ràng buộc |

**THỰC HÀNH:** điền `RISK_TIER` và `tier_reason`, `tier_obligations` cho use case của nhóm.

In [4]:
EU_TIERS = ['unacceptable', 'high', 'limited', 'minimal']

# THỰC HÀNH: điền ba giá trị dưới đây.
RISK_TIER = 'limited'            # chọn trong EU_TIERS
tier_reason = '… (lý do xếp mức: output đi vào đâu, ai chịu ảnh hưởng, mức tự động hóa)'
tier_obligations = '… (nghĩa vụ tương ứng: minh bạch / giám sát / tài liệu / đánh giá)'

assert RISK_TIER in EU_TIERS, f'RISK_TIER phải thuộc {EU_TIERS}'
eu_tier = pd.DataFrame([{
    'use_case': SELECTED_USECASE, 'risk_tier': RISK_TIER,
    'lý_do': tier_reason, 'nghĩa_vụ': tier_obligations,
}])
display(eu_tier.T.rename(columns={0: 'nội dung'}))
eu_tier.to_csv(OUTPUT_DIR / 'eu_risk_tier.csv', index=False)


,nội dung
use_case,RAG
risk_tier,limited
lý_do,"… (lý do xếp mức: output đi vào đâu, ai chịu ả..."
nghĩa_vụ,… (nghĩa vụ tương ứng: minh bạch / giám sát / ...


# 3. AI Risk Assessment — 5 nhóm
Mỗi nhóm rủi ro cần gắn với **một biện pháp kiểm soát cụ thể** (metric, guardrail, quy trình) — không dừng ở định nghĩa.

| Nhóm | Câu hỏi kiểm tra nhanh |
|---|---|
| **Accuracy/Hallucination** | Có test set chưa? Câu trả lời có nguồn? Được phép nói "không biết"? |
| **Fairness/Bias** | Đã đo lỗi theo nhóm chưa? Nhóm nào dễ bị thiệt? |
| **Privacy** | Dữ liệu nào bị cấm/cần masking? Có đồng ý chưa? |
| **Security** | Tool/model có quyền gì? Có input validation? Prompt injection? |
| **Accountability** | Ai duyệt? Ai chịu trách nhiệm? Log ở đâu? |

**THỰC HÀNH:** với mỗi nhóm, điền **ít nhất 1 rủi ro cụ thể trong use case** và **1 biện pháp kiểm soát cụ thể**. Thay các ô `'…'`.

In [5]:
RISK_GROUPS = ['Accuracy/Hallucination', 'Fairness/Bias', 'Privacy', 'Security', 'Accountability']

# THỰC HÀNH: điền hai cột cuối cho từng nhóm. Tránh để '…'.
risk_rows = [
    {'Nhóm': 'Accuracy/Hallucination', 'Rủi ro cụ thể trong use case': '…', 'Biện pháp kiểm soát': '…'},
    {'Nhóm': 'Fairness/Bias',          'Rủi ro cụ thể trong use case': '…', 'Biện pháp kiểm soát': '…'},
    {'Nhóm': 'Privacy',                'Rủi ro cụ thể trong use case': '…', 'Biện pháp kiểm soát': '…'},
    {'Nhóm': 'Security',               'Rủi ro cụ thể trong use case': '…', 'Biện pháp kiểm soát': '…'},
    {'Nhóm': 'Accountability',         'Rủi ro cụ thể trong use case': '…', 'Biện pháp kiểm soát': '…'},
]
risk_assessment = pd.DataFrame(risk_rows)
assert list(risk_assessment['Nhóm']) == RISK_GROUPS, 'Đủ và đúng thứ tự 5 nhóm.'
display(risk_assessment)

# Kiểm tra hoàn thiện (cảnh báo nếu còn '…').
chua_dien = risk_assessment[(risk_assessment == '…').any(axis=1)]['Nhóm'].tolist()
if chua_dien:
    print('Cảnh báo: các nhóm còn trống, hoàn thiện trước khi nộp:', chua_dien)
else:
    print('Đã điền đủ 5 nhóm.')
risk_assessment.to_csv(OUTPUT_DIR / 'risk_assessment_5groups.csv', index=False)


,Nhóm,Rủi ro cụ thể trong use case,Biện pháp kiểm soát
0,Accuracy/Hallucination,…,…
1,Fairness/Bias,…,…
2,Privacy,…,…
3,Security,…,…
4,Accountability,…,…


Cảnh báo: các nhóm còn trống, hoàn thiện trước khi nộp: ['Accuracy/Hallucination', 'Fairness/Bias', 'Privacy', 'Security', 'Accountability']


# 4. Explainability & Bias — hai demo nhỏ
Hai demo dưới chạy trên **dữ liệu mô phỏng**, không cần model thật, để thấy hai ý quan trọng:
1. **Bias theo nhóm:** accuracy tổng thể cao vẫn có thể che giấu bất công ở nhóm thiểu số.
2. **Giải thích kiểu LIME:** từ/đặc trưng nào đẩy dự đoán về một nhãn.

### Bước 4a — Demo bias: accuracy tổng thể che giấu lỗi theo nhóm
Dữ liệu mô phỏng: một classifier đo trên 2 nhóm site — **site lớn** (nhiều dữ liệu huấn luyện) và **site nhỏ** (ít dữ liệu). Xem accuracy tổng vs accuracy từng nhóm.

In [6]:
# Mô phỏng kết quả dự đoán: 1 = đúng, 0 = sai. Site lớn tốt hơn hẳn site nhỏ.
# (Số liệu giả để minh họa; nhóm có thể đổi tỷ lệ để thử.)
bias_eval = pd.DataFrame(
    [{'nhóm': 'site_lớn', 'correct': 1} for _ in range(90)] +
    [{'nhóm': 'site_lớn', 'correct': 0} for _ in range(10)] +
    [{'nhóm': 'site_nhỏ', 'correct': 1} for _ in range(12)] +
    [{'nhóm': 'site_nhỏ', 'correct': 0} for _ in range(8)]
)

acc_tong = bias_eval['correct'].mean()
acc_nhom = bias_eval.groupby('nhóm')['correct'].agg(['mean', 'count']).rename(
    columns={'mean': 'accuracy', 'count': 'số_mẫu'})
print(f'Accuracy TỔNG THỂ: {acc_tong:.2%}  (trông rất đẹp!)')
display(acc_nhom)
print('=> Site nhỏ bị thiệt rõ rệt dù accuracy tổng vẫn cao. Đây là bias bị che.')


Accuracy TỔNG THỂ: 85.00%  (trông rất đẹp!)


,accuracy,số_mẫu
nhóm,,
site_lớn,0.9,100
site_nhỏ,0.6,20


=> Site nhỏ bị thiệt rõ rệt dù accuracy tổng vẫn cao. Đây là bias bị che.


**THỰC HÀNH:**
- Trong use case của bạn, **nhóm nào** có vai trò như `site_nhỏ` (dễ bị bỏ sót)? Ví dụ: khu vực xa, thiết bị cũ, ca đêm, ngôn ngữ hiếm… …
- Bạn sẽ **đo lỗi theo nhóm** bằng metric nào (precision/recall theo phân khúc)? …
- Nếu phát hiện bias, hai biện pháp giảm thiểu bạn chọn là gì? …


### Bước 4b — Demo giải thích kiểu LIME (local explanation)
LIME giải thích **một dự đoán** bằng cách xem từ/đặc trưng nào đẩy về nhãn. Ở đây ta mô phỏng bằng một "model" tuyến tính đơn giản: mỗi từ khóa có trọng số đẩy về nhãn `Connectivity`. (Minh họa cơ chế; với model thật dùng thư viện `lime`/`shap`.)

> **Cảnh báo:** lời “giải thích” do LLM tự sinh có thể hợp lý nhưng không phản ánh đúng cơ chế thật; cần kiểm chứng bằng bằng chứng (grounding/citation), không tin suông. SHAP/LIME là công cụ hỗ trợ hiểu, không phải bằng chứng tuyệt đối về nhân quả.

In [7]:
# Trọng số từ khóa mô phỏng đẩy về nhãn 'Connectivity' (dương = đẩy về, âm = đẩy ra).
KEYWORD_WEIGHTS = {
    'mất kết nối': 0.9, 'timeout': 0.7, 'ping': 0.5, 'gateway': 0.4,
    'chậm': -0.3, 'độ trễ': -0.4, 'quạt': -0.6, 'cấu hình': -0.5,
}

def explain_like_lime(text, weights=KEYWORD_WEIGHTS):
    t = text.lower()
    contrib = {kw: w for kw, w in weights.items() if kw in t}
    score = sum(contrib.values())
    return score, dict(sorted(contrib.items(), key=lambda kv: -abs(kv[1])))

sample_ticket = 'Site HP06 không truy cập được gateway, nhiều lần timeout; ping thất bại.'
score, contrib = explain_like_lime(sample_ticket)
print('Ticket:', sample_ticket)
print(f'Score đẩy về Connectivity: {score:+.2f} => dự đoán:', 'Connectivity' if score > 0 else 'Khác')
print('Đóng góp từng từ (giảm dần độ lớn):')
display(pd.DataFrame([{'từ/đặc trưng': k, 'đóng góp': v} for k, v in contrib.items()]))


Ticket: Site HP06 không truy cập được gateway, nhiều lần timeout; ping thất bại.
Score đẩy về Connectivity: +1.60 => dự đoán: Connectivity
Đóng góp từng từ (giảm dần độ lớn):


,từ/đặc trưng,đóng góp
0,timeout,0.7
1,ping,0.5
2,gateway,0.4


**THỰC HÀNH:**
- Nếu use case là **classifier NLP/CV**: bạn giải thích bằng gì (SHAP/LIME, feature/từ/vùng ảnh quan trọng)? …
- Nếu use case là **LLM/RAG**: bạn giải thích bằng **grounding + citation** như thế nào (trích nguồn nào, cho phép "không đủ thông tin" không)? …
- Đổi `sample_ticket` thành một input thật của use case và chạy lại; giải thích có hợp lý không? …


# 5. AI Use-case Canvas
Một trang phải trả lời: vấn đề gì, dữ liệu gì, AI làm gì, đo ra sao, rủi ro gì, con người kiểm soát ở đâu. Giúp tránh nhảy thẳng vào model/framework.

**THỰC HÀNH:** điền 8 ô Canvas cho use case nhóm. Bảng dưới pre-fill từ use case đã chọn ở Phần 1 — sửa cho chính xác và điền các ô `'…'`.

In [ ]:
CANVAS_FIELDS = ['Problem', 'User', 'Data', 'AI task', 'Output', 'Metric', 'Risk/control', 'Human control']

# THỰC HÀNH: điền 8 ô. Một số ô đã pre-fill từ USECASES[SELECTED_USECASE].
canvas = {
    'Problem': uc['problem'],
    'User': f"… (ai dùng output? ai chịu ảnh hưởng? hành động sau output: {uc['action']})",
    'Data': uc['data'] + ' — phân loại & quyền: …',
    'AI task': uc['ai_task'],
    'Output': '… (nhãn/JSON/câu trả lời/bbox? có schema? đi vào đâu? ai sửa được?)',
    'Metric': '… (kỹ thuật: precision/recall/F1/faithfulness/latency + nghiệp vụ + bias theo nhóm)',
    'Risk/control': '… (rủi ro chính + guardrail: grounding/validation/filter/approval)',
    'Human control': f'… (Assist/Approve/Automate? người duyệt ở bước nào?)',
}
canvas_df = pd.DataFrame([{'Ô Canvas': k, 'Nội dung': canvas[k]} for k in CANVAS_FIELDS])
display(canvas_df)
canvas_df.to_csv(OUTPUT_DIR / 'usecase_canvas.csv', index=False)


# 6. Quyết định triển khai: go / pilot / not yet
| Mức | Điều kiện |
|---|---|
| **go** | Rủi ro thấp, metric đạt, dữ liệu ổn, guardrail rõ, hợp quy định |
| **pilot** | Có giá trị nhưng cần thử phạm vi hẹp, người giám sát chặt |
| **not yet** | Dữ liệu thiếu, rủi ro cao, chưa có owner hoặc metric/pháp lý chưa rõ |

Ngoài ra, chọn **mức tự động hóa** phù hợp: **Assist** (AI gợi ý, người quyết định) / **Approve** (AI chuẩn bị, người duyệt) / **Automate** (hệ thống tự hành động).

**THỰC HÀNH:** điền `DECISION`, `AUTOMATION_LEVEL`, `decision_reason`, `human_control_point`.

In [ ]:
DECISIONS = ['go', 'pilot', 'not yet']
AUTOMATION_LEVELS = ['Assist', 'Approve', 'Automate']

# THỰC HÀNH: điền bốn giá trị.
DECISION = 'pilot'
AUTOMATION_LEVEL = 'Assist'
decision_reason = '… (lý do dựa trên risk assessment, metric, dữ liệu, pháp lý)'
human_control_point = '… (con người can thiệp ở bước nào: thiết kế nhãn / duyệt output rủi ro cao / xử exception / review log)'

assert DECISION in DECISIONS and AUTOMATION_LEVEL in AUTOMATION_LEVELS
decision = pd.DataFrame([{
    'use_case': SELECTED_USECASE, 'decision': DECISION,
    'automation_level': AUTOMATION_LEVEL,
    'lý_do': decision_reason, 'human_control': human_control_point,
}])
display(decision.T.rename(columns={0: 'nội dung'}))
decision.to_csv(OUTPUT_DIR / 'deployment_decision.csv', index=False)


### Xuất markdown summary
Gộp tất cả phần thành một file markdown để nộp/trình bày. Chạy sau khi đã điền xong các cell trên.

In [8]:
def df_to_md(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

lines = []
lines.append('# Lab Bài 2 — Responsible AI & AI Risk Assessment')
lines.append(f'**Use case:** {SELECTED_USECASE} — {uc["ten"]}\n')
lines.append('## 1. Phân loại dữ liệu'); lines.append(df_to_md(data_class) + '\n')
lines.append('## 2. Mức rủi ro (EU AI Act)')
lines.append(f'- **Tier:** {RISK_TIER}\n- **Lý do:** {tier_reason}\n- **Nghĩa vụ:** {tier_obligations}\n')
lines.append('## 3. AI Risk Assessment (5 nhóm)'); lines.append(df_to_md(risk_assessment) + '\n')
lines.append('## 4. Explainability & Bias')
lines.append(f'- Accuracy tổng (demo): {acc_tong:.2%}; site_nhỏ thấp hơn rõ rệt → bias bị che.\n')
lines.append('## 5. AI Use-case Canvas'); lines.append(df_to_md(canvas_df) + '\n')
lines.append('## 6. Quyết định')
lines.append(f'- **Decision:** {DECISION}\n- **Automation:** {AUTOMATION_LEVEL}\n- **Lý do:** {decision_reason}\n- **Human control:** {human_control_point}\n')
summary_md = '\n'.join(lines)
(OUTPUT_DIR / 'lab2_summary.md').write_text(summary_md, encoding='utf-8')
print(summary_md)
print('\nĐã lưu tại:', (OUTPUT_DIR / 'lab2_summary.md').resolve())


NameError: name 'canvas_df' is not defined

# 7. Peer review + Debrief bắt buộc
### Peer review (đổi chéo nhóm)
Đổi notebook với nhóm khác, mỗi người điền nhận xét vào `peer_review` dưới.

### Câu hỏi debrief bắt buộc (trả lời ngay trong cell markdown này)
1. Use case này có thể **sai hoặc gây bất công** cho nhóm nào? …
2. Dữ liệu nào **bị cấm/cần masking** theo quy định Tập đoàn và pháp luật VN? …
3. Làm sao **giải thích quyết định** của model cho người dùng/kiểm toán (SHAP/LIME/grounding)? …
4. **Ai chịu trách nhiệm cuối cùng** và con người kiểm soát ở bước nào? …


In [ ]:
# THỰC HÀNH: nhóm review điền các ô '…'.
peer_review = pd.DataFrame([
    {'Tiêu chí': 'Phân loại dữ liệu hợp lý?', 'Nhận xét': '…'},
    {'Tiêu chí': 'Risk assessment đủ & cụ thể?', 'Nhận xét': '…'},
    {'Tiêu chí': 'Bias đã đo theo nhóm?', 'Nhận xét': '…'},
    {'Tiêu chí': 'Quyết định có căn cứ?', 'Nhận xét': '…'},
])
display(peer_review)
peer_review.to_csv(OUTPUT_DIR / 'peer_review.csv', index=False)
print('Đã lưu các bảng tại:', OUTPUT_DIR.resolve())


## Checklist hoàn thành
- [ ] Đã chọn use case và điền bảng phân loại dữ liệu (Public/Internal/Confidential/Restricted).
- [ ] Đã xếp mức rủi ro EU AI Act kèm lý do.
- [ ] Bảng AI Risk Assessment đủ 5 nhóm, mỗi nhóm có rủi ro + biện pháp cụ thể (không còn '…').
- [ ] Đã chạy 2 demo (bias theo nhóm, giải thích kiểu LIME) và ghi nhận xét.
- [ ] AI Use-case Canvas đủ 8 ô.
- [ ] Quyết định go/pilot/not yet + mức tự động hóa + lý do.
- [ ] Đã xuất `lab2_summary.md` và trả lời 4 câu debrief.
- [ ] Peer review đã điền.
- [ ] Không để credential/PII thật trong notebook.

**Nộp:** `Module3_Lab2_<ten_nhom>.ipynb` có output. Trong Colab: **File → Download → Download .ipynb**.

## Nguồn
Thiết kế theo slide **“AI Engineer – Module 3, Bài 2: Responsible AI, Explainable AI, kiểm soát Bias, bảo mật dữ liệu và đánh giá rủi ro”** và brief **“Lab Bài 2: Responsible AI & AI Risk Assessment”** do giảng viên cung cấp. Khung tham chiếu: Microsoft Responsible AI, Google AI Principles, EU AI Act, NIST AI RMF (Govern–Map–Measure–Manage). Dữ liệu trong 2 demo là mô phỏng cho mục đích giảng dạy.
